<a href="https://colab.research.google.com/github/dzianismr/GetPKParameterOpenFDA/blob/develop/OpenFDA_getPK_withQC_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This R script represents a simple LLM-based agentic workflow which **extracts, tabulates and QC PK information from the FDA labels**. The key steps are:
- retrieval of relevant labels via OpenFDA API;
- LLM-based extraction of PK information;
- Two level LLM-based QC of extracted PK information:
-- check of tabulated PK line-by-line;
-- chcek at for missing infomation at the compound level;

The script uses OpenAI gpt-4o LLM, therefore OPENAI_API_KEY needs to provided, alternatively a different LLM can be used.

In [ ]:
#set up env

##  instal
!!pip install litellm #
from litellm import completion # Python interface to LLMs

# import
import requests # supports API usage in Python
import json # JSON Parser for Python

# standard data wrangling packages
import pandas as pd
from typing import List, Dict
import time


In [ ]:
# set llm access
# for this to work "secrets" has to be set up in google colab
import os
from google.colab import userdata

# Set up LLM access
api_key = userdata.get('OpenAIAPI')
os.environ['OPENAI_API_KEY'] = api_key

OPENAI_API_KEY = userdata.get('OpenAIAPI')

In [ ]:
# Retrieve PK data from OpenFDA API
# n labels containing via pediatric PK information for monoclonal antibodies will be retrieved

# file name
DRUG = "paediatric_mab"
n = 3

url = "https://api.fda.gov/drug/label.json?search=openfda.generic_name:*mab+AND+pharmacokinetics:pediat*&limit="+ str(n)


# Make the GET request
response = requests.get(url)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()
    print("API request successful!")
    # Display the first 3 results for demonstration
    if 'results' in data and len(data['results']) > 0:
        print("First 3 results:")
        for i, result in enumerate(data['results'][:3]):
            print(f"\nResult {i+1}:")
            print(json.dumps(result, indent=2))
    else:
        print("No results found or 'results' key missing.")
else:
    print(f"API request failed with status code: {response.status_code}")
    #print(f"Response: {response.text}") # uncomment to print json output

In [ ]:
# define helper function to interact with LLM
def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""
    response = completion(
        model="openai/gpt-4o",
        messages=messages,
        max_tokens=16384
    )
    return response.choices[0].message.content

# prompts
message1 ={"role": "system", "content": "You an experienced pharmacokineticist with attention to the detail. You specialize in extracting PK parameters from the text. You always return results in JSON format."}
message2 ={"role": "user", "content":   "The following text was extracted from FDA labelling document. Extract reported PK parameters e.g. CL (clearance), Vd (volume of distribution), AUC (Caverage), Cmin (Ctrough), Cmax also extract uncertainty of the estimate as well as units. Extracted parameters should be reported in tabular view. Table columns should include: drug, dose, population, parameter, value, uncertainty_value, uncertainty_type, unit."}

messageQC1 ={"role": "user", "content":   "The following pharmacokinetic parameter and metadata were extracted from the PK section of the FDA labelling document. Perform quality control of the extracted PK parameter. Specifically, confirm numerical accuracy of the extracted PK parameter, its uncertainty and metadata describing unit of the parameter value, dose and population it was extracted from. Response should include two and only two values: QC: 1 = passed, 0 = failed. QC_comment: empty string if QC=1, a short description of identified error if QC=0."}
messageQC2 ={"role": "user", "content":   "The following summary table in JSON format contains pharmacokinetic parameters and metadata extracted from the PK section of the FDA labelling document. Check if all PK parameters, for which numerical values are available in the FDA labelling document, specifically CL (clearance), Vd (volume of distribution), AUC (Caverage), Cmin (Ctrough), Cmax are present in the summary table. If an additional PK parameter is mentioned, however it's numerical value is not reported - ignore it. Response should include two and only two values COMPLETE and COMMENT. COMPLETE: is set to 1 if summary table is complete and to 0 if summary table is not complete. COMMENT: contains empty string if COMPLETE=1, if COMPLETE=0 COMMENT lists PK parameters missing in the summary table, specifically PK parameters, their values and population. Importantly, COMPLETE should be a single string."}

In [ ]:
# helper function llm output processing
def parse_llm_response(resp_raw):
    # Clean JSON response
    resp_cur = resp_raw.strip().replace('```json', '').replace('```', '')
    resp_cur = json.loads(resp_cur)
    #resp_cur = pd.DataFrame([resp_cur])

    return(resp_cur)


In [ ]:
# helper function correctness check
def check_correctness_f(df_pk_cur, pk_text):

    # Convert each dataframe row into one JSON object.
    # One line corresponds to one PK parameter.
    json_pk1_str = json.dumps(
        df_pk_cur.iloc[0].to_dict(),
        indent=2
    )

    # Create prompt by combining user instruction with the pk section and
    # extracted PK parameters
    updated_messageQC1 = messageQC1.copy()
    updated_messageQC1['content'] = (
        messageQC1.get('content', '')
        + " Extracted PK parameters in JSON format: "
        + json_pk1_str
        + ". PK Section of the labelling document: "
        + pk_text
    )

    messages = [
        message1,
        updated_messageQC1
    ]

    # Submit prompt and get response from the LLM
    raw_response_QC = generate_response(messages)

    return str(raw_response_QC)


# helper function completeness check
def check_completness_f(temp_df, pk_text):

    # Select relevant columns
    pk_df = temp_df[
        ["dose", "population", "parameter", "value"]
    ]

    # Convert each dataframe row into one JSON object.
    # One line corresponds to one PK parameter.
    json_pk1_str = "\n".join(
        json.dumps(record, ensure_ascii=False)
        for record in pk_df.to_dict(orient="records")
    )

    # Create prompt by combining user instruction with the PK section
    updated_messageQC2 = messageQC2.copy()
    updated_messageQC2['content'] = (
        messageQC2.get('content', '')
        + " Extracted PK parameters in JSON format: "
        + json_pk1_str
        + ". PK Section of the labelling document: "
        + pk_text
    )

    messages = [
        message1,
        updated_messageQC2
    ]

    # Submit prompt and get response from the LLM
    raw_response_QC = generate_response(messages)

    # Display the QC result
    print(raw_response_QC)

    return str(raw_response_QC)

In [ ]:
all_pk_dfs = []
all_pk_qc  = []

# iterate through the list of results, compound by compound
if 'results' in data and len(data['results']) > 0:
    print(f"Processing {len(data['results'])} documents...")

    for idx, result in enumerate(data['results']):

        # 1. get Drug Name (Brand or Generic)
        brand_name   = result.get('openfda', {}).get('brand_name', ['Unknown'])[0]
        generic_name = result.get('openfda', {}).get('generic_name', ['Unknown'])[0]
        current_drug_label = generic_name

        print(f"--- Processing Result {idx+1}: {current_drug_label} ---")

        # 2. Extract Pharmacokinetics section
        pk_text_list = result.get("pharmacokinetics") or result.get("12.3 Pharmacokinetics")

        if not pk_text_list:
            print(f"No PK section found for {current_drug_label}. Skipping.")
            continue

        pk_text = " ".join(pk_text_list)

        # 3. Prepare and call LLM
        # create prompt by combining user instruction with the pk section
        updated_message2 = message2.copy()
        updated_message2['content'] = updated_message2.get('content', '') + pk_text

        messages = [
            message1,
            updated_message2
        ]


        try:
            # submit prompt and get response from the LLM
            raw_response = generate_response(messages)
            temp_df      = parse_llm_response(raw_response)
            temp_df      = pd.DataFrame(temp_df)
            print(f"Successfully extracted {len(temp_df)} parameters.")

            # completeness check
            print("Run completness check")

            qc_cur_raw = check_completness_f(temp_df, pk_text)
            qc_cur     = parse_llm_response(qc_cur_raw)
            qc_cur     = pd.DataFrame([qc_cur])

            qc_cur = qc_cur.assign(
                source_brand_name=brand_name,
                source_generic_name=generic_name
            )

            # correctness check
            print("Run accuracy check")
            for j in range(len(temp_df)):
              df_pk_cur = temp_df.iloc[[j]]
              raw_qc = check_correctness_f(df_pk_cur, pk_text)
              temp_qc = parse_llm_response(raw_qc)
              temp_qc = pd.DataFrame([temp_qc])

            # store dataframes
            if not temp_df.empty:
                temp_df['source_brand_name']   = brand_name
                temp_df['source_generic_name'] = generic_name
                all_pk_dfs.append(temp_df)

            if not qc_cur.empty:
                qc_cur['source_brand_name']   = brand_name
                qc_cur['source_generic_name'] = generic_name
                all_pk_qc.append(qc_cur)

        except Exception as e:
            print(f"Error processing {current_drug_label}: {e}")


In [ ]:
# 5. Combine all results into a single Long Format table

# Ensure the output directory exists
output_dir = 'out_getPK'
os.makedirs(output_dir, exist_ok=True)

if all_pk_dfs:
    final_long_df = pd.concat(all_pk_dfs, ignore_index=True)
    display(final_long_df)

    final_long_qc = pd.concat(all_pk_qc, ignore_index=True)
    display(final_long_qc)

    # Export the consolidated PK table
    final_export_path = os.path.join(output_dir, f"all_{DRUG}_labels_PK_long.csv")
    final_long_df.to_csv(final_export_path, index=False)
    print(f"\nFinal consolidated table saved to: {final_export_path}")

    # Export the consolidated qc table
    final_export_path = os.path.join(output_dir, f"all_{DRUG}_labels_QC.csv")
    final_long_qc.to_csv(final_export_path, index=False)
    print(f"\nFinal consolidated table saved to: {final_export_path}")
else:
    print("No PK data was extracted from any of the documents.")